# 01 - Sanity check

Run this once the trainax integration in `rollout_error.runner.run_cell` is wired. It checks that the pieces line up **before** committing to the full sweep.

Nothing here should train for long: one scenario, one architecture, one seed, short horizons.

## 1. Imports resolve, configs load

- `import rollout_error` and submodules succeed.
- `rollout_error.sweep.enumerate_jobs()` returns a non-empty list and every job has all six axes populated.
- `rollout_error.runner.resolve_config(job)` merges the four YAML layers without a `KeyError`.

## 2. Weighting strategy behaves as specified

- `uniform` -> all-ones weight vector.
- `discounted` -> `gamma ** k`; check gamma on both sides of 1.0 (0.8 down-weights the tail, 1.25 up-weights it).
- `normalized` -> weights track `1 / EMA(per-step error)`; EMA state is returned, not mutated in place.
- `weighted_rollout_loss(pred, target, mode='uniform')` equals the arithmetic mean of the per-step residual norms.

## 3. One APEBench cell end to end

- Instantiate `diff_adv` at the configured difficulty, `Conv;26;10;relu`, `train_config='one'`, one seed.
- Roll out over the test horizon; build `pred_traj` / `target_traj`.
- `rollout_error.metrics.rollout_report(...)` returns per-step nRMSE and Pearson curves of length `test_temporal_horizon`.
- `rows_from_report` produces one row per step with the documented schema; `append_rows` writes a parquet that reads back identically.

## 4. Null-control smell test

- Run `diff_diff` (the null control) with `train_mode='sup'` and `train_mode='wsup'` (discounted, gamma=0.9), one seed.
- The two per-step nRMSE curves should sit on top of each other and both stay low/flat: pure diffusion damps error regardless of loss weighting.
- If they diverge here, stop and debug before running the sweep.